<a href="https://colab.research.google.com/github/EricChen356/video-background-remover/blob/main/SAM3_Video_Background_Remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM 3 Video Background Remover  ·  green-screen / alpha / transparent

SAM 3 -> MatAnyone2 pipeline


Keeps the **foreground subject(s)** and replaces the background with a green screen,
holding up on **fine hair edges** and staying **temporally stable** across scene cuts.

**Pipeline (per video):**
1. **Probe** metadata (fps, frames, resolution, audio).
2. **Scene-cut detection** with **PySceneDetect** (`AdaptiveDetector` + `ThresholdDetector` for fades).
3. **Per shot** (fresh SAM3 session so tracking memory never smears across a cut):
   - **SAM 3** (HF `transformers`, *text* prompt `"person"`) → per-frame instance masks + stable `object_id`s.
   - **Foreground selection** via **Depth Anything V2** (median depth inside each mask) + geometry tie-breakers.
   - **MatAnyone** mask-guided matting, seeded from the coarse mask → soft alpha with hair detail.
     Re-seeded when a new subject becomes foreground mid-shot.
4. **Composite** in float: `out = fg*alpha + green*(1-alpha)`.
5. **Reassemble** at original fps, **mux audio**, write atomically, resume-skip.

> Verified against the docs before coding: SAM3 video uses **text prompts only** (no boxes);
> `facebook/sam3.1` is weights-only with no `transformers` integration, so the transformers path
> uses `facebook/sam3`. Depth Anything V2 outputs **relative inverse depth** (larger = nearer).

In [ ]:
#  ──── INSTALL DEPENDENCIES ──────────────────────────────────────────────────────────────────────
import os
from google.colab import userdata
!git clone https://github.com/pq-yang/MatAnyone2.git
%cd MatAnyone2
!pip install -e . -q

!pip install git+https://github.com/facebookresearch/sam3.git -q

!pip install ultralytics -q

!pip install torch torchvision -q

!pip install "imageio>=2.33,!=2.35.0" "huggingface-hub>=1.5.0,<2.0" -q

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print('\n✅ All dependencies installed.')


In [ ]:
#  ──── INSTALL MODEL WEIGHTS ────────────────────────────────────────────────
import os, requests, tqdm

MA2_PATH = 'pretrained_models/matanyone2.pth'
MA2_URL  = 'https://github.com/pq-yang/MatAnyone2/releases/download/v1.0.0/matanyone2.pth'

if os.path.exists(MA2_PATH):
    os.remove(MA2_PATH)
    print('Removed corrupt checkpoint')

os.makedirs('pretrained_models', exist_ok=True)

response = requests.get(MA2_URL, stream=True)
total = int(response.headers.get('content-length', 0))

with open(MA2_PATH, 'wb') as f, tqdm.tqdm(
    desc='matanyone2.pth',
    total=total,
    unit='B',
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        bar.update(len(chunk))

print(f'Downloaded: {os.path.getsize(MA2_PATH) / 1e6:.1f} MB')

In [ ]:
# ──── USER CONFIG ────────────────────────────────────────────────────────────────────────
# Everything you normally tune lives here. Defaults are conservative and work well
# for typical single/multi-person footage — change values, not the cells below.
import os as _os
import numpy as np

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_DIR   = '/content/videos'           # folder of input videos (processed as a batch)
OUTPUT_DIR  = '/content/matanyone2_out'   # mattes / alpha / debug masks are written here

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_MODE = 'greenscreen'   # 'greenscreen' | 'alpha' | 'transparent'
GREEN       = np.array([120, 255, 152], dtype=np.float32) / 255.0   # key colour for greenscreen

# ── Performance ───────────────────────────────────────────────────────────────
MAX_SIZE    = 0               # 0 = full resolution; else matte at this short-side size

# -- Encoding -----------------------------------------------------------------
WRITE_ALPHA_VIDEO = True
ENCODER_PRESET    = 'veryfast'
ENCODER_CRF       = 18

# ── Matting quality (MatAnyone 2) ─────────────────────────────────────────────
N_WARMUP_STATIC = 20
N_WARMUP_MOTION = 20
R_ERODE     = 0       # optional: erode the seed a few px so matting starts strictly
                      # inside the subject — use if the matte still bleeds onto an
                      # adjacent wall/floor after lowering R_DILATE (risks thin parts).
R_DILATE    = 8       # seed mask is dilated this many px before matting. SMALL keeps the
                      # matte tight to the subject; LARGE (e.g. 20) grows the seed past
                      # the body onto nearby background (a wall / the floor shadow behind
                      # the subject), which MatAnyone then locks into memory and
                      # propagates through the whole shot. Raise only if fine edges (hair,
                      # fingers) get cut on the first frame.

# ── Detection thresholds ──────────────────────────────────────────────────────
YOLO_CONF     = 0.3   # detection confidence floor (people AND held props)
CUT_THRESHOLD = 0.15  # scene-cut sensitivity (lower = more cuts)

# BOX_SCORE_RATIO: YOLO person boxes are scored by size + centrality + confidence.
#   Only boxes scoring >= this fraction of the best box are passed to SAM 3.
#   Lower this (e.g. 0.1) if YOLO detects a person who is not being masked.
BOX_SCORE_RATIO = 0.3

# PERSON_REL_SIZE_MIN / PERSON_CENTER_KEEP: relative-size background gate. After the
#   prominence score above, every YOLO person box is compared to the LARGEST person in
#   the frame (the dominant subject). A box is dropped as a background bystander only when
#   it is BOTH much smaller than that subject AND off to the side — the signature of a
#   passer-by — so SAM 3 is never prompted there and the concept mask can't grab it:
#     - PERSON_REL_SIZE_MIN — size ratio (LINEAR: box scale / biggest-box scale) below
#       which a box counts as 'much smaller'. 0.5 = 'more than twice as small'. Raise to
#       drop more aggressively; lower to keep more distant people.
#     - PERSON_CENTER_KEEP — a small box at least this central (centrality 0..1; 1 = frame
#       centre, ~0.5 = its centre at the edge) is kept anyway, so a distant lead framed in
#       the middle survives. Lower it to also drop small central figures.
#   The gate is RELATIVE, so a wide group where everyone is a similar (even small) size
#   keeps everyone — no one is 'twice as small' as the rest — while a lone figure that is
#   both small and pushed to the edge is removed. The most-prominent box is never dropped.
#   Set PERSON_REL_SIZE_MIN = 0.0 to disable the gate entirely.
PERSON_REL_SIZE_MIN = 0.6
PERSON_CENTER_KEEP  = 0.7

# SAM_MASK_THRESHOLD: threshold on the CONCEPT head's mask logits — the clean,
#   high-quality subject silhouette. 0.5 is the original tuned value; lower toward
#   0.0 for a slightly more inclusive subject edge, raise for a tighter one.
SAM_MASK_THRESHOLD = 0.5

# ── Concept-head reliability (why SAM 3 may not segment a person YOLO found) ──
# SAM 3's concept head is a CONFIDENCE-GATED detector — it only returns a mask when it
# is confident the region holds the concept, so a hard subject (back turned, low
# contrast, occluded) can be dropped entirely even though YOLO detected it. These knobs
# make it fire reliably so a clean silhouette exists to gate the interactive head.
#
# SAM_DETECT_THRESHOLD: concept-detector confidence gate (sigmoid prob). Lower it so
#   borderline subjects are still segmented; None = leave the model default. Raise it if
#   spurious extra detections start appearing.
SAM_DETECT_THRESHOLD = 0.2
# CONCEPT_TEXT_DETECT: also run an exhaustive text ("person") detection and union the
#   instance matching each YOLO box (by IoU) into the concept silhouette. Text detection
#   is SAM 3's best-tuned path and recovers people the bare box-geometric prompt misses.
CONCEPT_TEXT_DETECT = True
CONCEPT_TEXT_PROMPT = 'person'
CONCEPT_MATCH_IOU   = 0.3    # min YOLO-box vs concept-instance IoU to treat as same subject

# FORCE_INTERACTIVE: also run SAM 3's forceful interactive box->mask head and union it
#   with the concept mask. It has NO concept gate, so it always fires — recovering held
#   objects AND a subject the concept head missed — but it also fills in background the
#   subject wraps around. INTERACTIVE_GATE_BG (below) strips that fill back out.
#   Set False for concept-only behaviour (held objects / missed subjects may be lost).
FORCE_INTERACTIVE = True

# INTERACTIVE_MASK_THRESHOLD: threshold on the interactive head's logits (0.0 is its
#   natural boundary). Lower to pull in more of a held object; raise to be stricter.
INTERACTIVE_MASK_THRESHOLD = 0.0

# BOX_CLIP_MARGIN: the interactive box->mask head can occasionally leak onto a
#   high-contrast region far from the prompt box. We keep only mask blobs that
#   overlap the box expanded by this fraction on every side, which drops far-field
#   leaks while keeping the WHOLE subject blob (it is never cut at the box edge).
#   Raise it if a held item sits just outside the detected person box.
BOX_CLIP_MARGIN = 0.10

# INTERACTIVE_GATE_BG / GATE_BG_MARGIN: the interactive head is inclusive — beyond the
#   person it grabs missed limbs, held props, the background a body wraps around (the
#   arm-on-hip <-> torso gap) AND distinct objects the subject merely stands next to (a
#   brightly-coloured car). This gate uses APPEARANCE to keep only what belongs to the
#   subject: an interactive-added region is kept ONLY when its colour is clearly more
#   SUBJECT-like than background-like (a missed limb / same clothing); everything else is
#   dropped. It is deliberately drop-by-default, so a distinct third object (a car the
#   person stands beside) is removed rather than kept — this is what stops background
#   leaking into the mask. YOLO prop points are always kept. GATE_BG_MARGIN (0..1) is how
#   decisively subject-like a region must be to survive — higher = stricter (drops more),
#   lower = keeps more. Set INTERACTIVE_GATE_BG = False to union everything (leaks bg).
INTERACTIVE_GATE_BG = True
GATE_BG_MARGIN = 0.05

# RECOVER_HELD_OBJECTS: SAM's interactive head can miss a held object it doesn't
#   recognise (e.g. a lav-mic transmitter), and YOLO may never detect it, so it ends up
#   keyed out. This adds back regions INSIDE a person box that the masks missed when they
#   (a) are surrounded by the subject (held against the body) AND (b) do NOT look like the
#   scene background — i.e. a distinct object the person is holding. Background showing
#   through the body and regions the subject doesn't surround are left out. (Inverse of
#   the bg gate; shares its appearance model.)
#   OFF by default: between nearby people it added spurious BOX-SHAPED regions to the
#   seed — the unmasked pocket between two dancers is bordered by bodies and doesn't
#   match the background colour, so it passed the surround+colour tests and was added,
#   clipped to the rectangular person box. It also never reliably recovered a real
#   prop. The interactive head already grabs items held against the body, so the final
#   seed is exactly  concept OR interactive(gated), minus dropped,  without it.
RECOVER_HELD_OBJECTS  = False
RECOVER_SURROUND_FRAC = 0.6   # min fraction of a region's border that must be the subject
RECOVER_BG_MAXPROB    = 96    # 0..255; regions more background-like than this are NOT recovered
RECOVER_MAX_FRAC      = 0.02  # ignore recovered regions larger than this frac of the frame — a
                              # held prop is tiny; this stops a nearby background object (e.g. a
                              # car between two people) being pulled in as a giant 'held item'.

# ── Held-object handling ──────────────────────────────────────────────────────
# The interactive box→mask already grabs items held against the body. For props
# YOLO can also recognise, we additionally drop a positive SAM 3 point on them so
# they are pulled into the subject's mask instead of being keyed out as background.
# MANUAL_PROP_POINTS: the reliable way to segment a held prop YOLO can't detect (a
#   handheld / lav mic). Deciding "held prop vs background" automatically from a person box
#   is the exact ambiguity that caused earlier false positives (the red car, the box
#   between dancers) — it is NOT safely solvable by a heuristic. Instead give SAM ONE
#   positive point ON the prop: its interactive head then segments the prop cleanly, and
#   the background gate always KEEPS a region that carries a prop point, so nothing else in
#   the pipeline changes and no new background can leak in.
#   Format:  { 'video_basename': { shot_number: [(x_frac, y_frac), ...] } }
#     - x_frac / y_frac are 0..1 of the frame (read them off the diagnostic's seed frame:
#       e.g. a mic ~42% across and ~63% down -> (0.42, 0.63)).
#     - shot_number matches the diagnostic's "Shot N" label.
#     - a point outside every person box is ignored.
#   Leave the dict empty to disable. Add points, re-run the diagnostic to confirm the prop
#   turns red in cols 2/4, then run the pipeline.
MANUAL_PROP_POINTS = {
    # 'my_interview_clip': {1: [(0.42, 0.63)]},   # point on the mic in shot 1
}

MASK_HELD_OBJECTS    = True
HELD_OBJECT_MAX_FRAC = 0.25   # ignore "props" larger than this fraction of frame area
                              # (avoids grabbing background furniture inside the box)
# COCO class ids for small objects a person is likely to hold. (A microphone is not a
# COCO class, but hand-held recorders/mics often register as cell phone / remote.)
HELD_OBJECT_CLASSES = {
    24, 25, 26, 28,        # backpack, umbrella, handbag, suitcase
    32,                    # sports ball
    39, 40, 41,            # bottle, wine glass, cup
    43, 44,                # knife, spoon
    63, 64, 65, 66, 67,    # laptop, mouse, remote, keyboard, cell phone
    73, 74, 76, 77, 79,    # book, clock, scissors, teddy bear, toothbrush
}

# FILL_MASK_HOLES / MAX_HOLE_FRAC: flood-fill ONLY small enclosed holes (speckle, minor
#   mask gaps) up to MAX_HOLE_FRAC of the subject area. A large genuine negative-space
#   region — e.g. the gap between an arm-on-hip and the torso — is LEFT OPEN so the
#   background shows through. Set MAX_HOLE_FRAC = 1.0 to fill every hole, or
#   FILL_MASK_HOLES = False to never fill. OFF by default so the final seed is EXACTLY
#   concept OR interactive(gated), minus dropped — nothing synthesised. The R_DILATE
#   step before matting already closes pinholes this small; set True (MAX_HOLE_FRAC
#   e.g. 0.01) only if you see tiny transparent specks inside a solid body in the matte.
FILL_MASK_HOLES = False
MAX_HOLE_FRAC   = 0.01

# ── Gap fill (fill every hole, then subtract background LAST) ──────────────────
# GAP_FILL: after the concept + interactive heads run, FILL every fully-enclosed hole in the
#   mask (a pocket the body wraps around), then — as the final step — RE-OPEN only the holes
#   whose colour matches the scene background. A hole that looks DISTINCT from the background
#   (a dark mic the concept head skipped, held against the body) is KEPT as foreground; a hole
#   that shows the background (the arm-on-hip <-> torso gap, the floor between two dancers) is
#   re-opened. Topology can't tell a prop from a background gap, so appearance decides, and
#   because the re-open is the LAST step nothing can undo it. A hole with a prop point is
#   always kept. This replaces the old size-based hole fill (FILL_MASK_HOLES). False disables.
GAP_FILL      = True
GAP_BG_MARGIN = 0.05   # how decisively background-like a filled hole must be to re-open it
                       #   (0..1; higher keeps more holes FILLED/foreground, lower re-opens more)
GAP_MIN_AREA  = 64     # ignore holes smaller than this many px (speckle just stays filled)
GAP_MAX_FRAC  = 0.05   # a hole bigger than this frac of the frame is structural background —
                       #   left open, never treated as a held prop

# ── Troubleshooting ───────────────────────────────────────────────────────────
SAVE_SEED_MASKS = True

VIDEO_EXTENSIONS = ('*.mp4', '*.mov', '*.avi', '*.mkv')

_os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded.')


In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# TF32 matmuls on Ampere+ GPUs — a near-free speedup with no visible quality change.
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ── MatAnyone 2 (video matting) ───────────────────────────────────────────────
# Loaded and kept in float32 — do NOT wrap in global bfloat16 autocast, as that
# degrades matting quality for borderline pixels (fine hair, held objects, etc.).
from hydra.core.global_hydra import GlobalHydra
from matanyone2.utils.get_default_model import get_matanyone2_model

GlobalHydra.instance().clear()
matanyone2_model = get_matanyone2_model(MA2_PATH, device)
matanyone2_model = matanyone2_model.float()
print('✅ MatAnyone 2 loaded')

# ── SAM 3 (box / text → pixel-precise mask) ───────────────────────────────────
# Built with enable_inst_interactivity=True so the SAM 1-style interactive predictor
# (sam3_model.inst_interactive_predictor) is loaded from the same facebook/sam3
# checkpoint. That predictor takes a box and segments whatever is INSIDE it —
# straight to the segmentation head, with NO detector / concept re-analysis and NO
# confidence gate — which is what keeps hand-held objects (mics, phones) in the
# foreground instead of being keyed green. Sam3Processor's CONCEPT head makes
# the clean subject silhouette; the interactive head is UNIONED in to force-
# segment held objects the concept head drops. SAM 3 weights are bfloat16;
# autocast is applied
# locally around each SAM 3 call so MatAnyone2 (float32) is unaffected.
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

sam3_model = build_sam3_image_model(device=device, eval_mode=True,
                                    enable_inst_interactivity=True)
sam3_predictor   = Sam3Processor(sam3_model, device=device)   # text-prompt fallback
sam3_interactive = sam3_model.inst_interactive_predictor      # box→mask head
# NOTE: call it via sam3_model.predict_inst(state, box=...) — the predictor
# reuses the main backbone features cached by Sam3Processor.set_image(); it has
# no backbone of its own, so inst_interactive_predictor.set_image() would fail.
assert sam3_interactive is not None, \
    'inst_interactive_predictor is None — rebuild with enable_inst_interactivity=True'
print('✅ SAM 3 loaded (interactive box→mask enabled)')

# ── YOLOv8n (person + held-prop detector → seed boxes/points for SAM 3) ────────
from ultralytics import YOLO
yolo_model = YOLO('yolov8n.pt')
print('✅ YOLOv8n loaded')


In [ ]:
import time
import torch.nn.functional as F
import imageio
from tqdm import tqdm
from contextlib import contextmanager, nullcontext
from collections import OrderedDict
from matanyone2.inference.inference_core import InferenceCore
from matanyone2.utils.inference_utils import gen_dilate, gen_erosion

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 0 — STEP TIMER
# ─────────────────────────────────────────────────────────────────────────────
class StepTimer:
    ORDER = ['decode', 'scene_cut', 'yolo', 'sam', 'mask_debug',
             'matting_pre', 'matting_warmup', 'matting_infer', 'matting_post',
             'composite', 'encode']

    def __init__(self, sync_cuda: bool = True):
        self.totals = OrderedDict()
        self.counts = OrderedDict()
        self.sync_cuda = bool(sync_cuda) and torch.cuda.is_available()

    @contextmanager
    def track(self, name: str):
        if self.sync_cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        try:
            yield
        finally:
            if self.sync_cuda:
                torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            self.totals[name] = self.totals.get(name, 0.0) + dt
            self.counts[name] = self.counts.get(name, 0) + 1

    def merge(self, other: 'StepTimer'):
        for k, v in other.totals.items():
            self.totals[k] = self.totals.get(k, 0.0) + v
            self.counts[k] = self.counts.get(k, 0) + other.counts[k]

    def report(self, title: str = 'Timing breakdown', n_frames: int | None = None):
        keys  = [k for k in self.ORDER if k in self.totals]
        keys += [k for k in self.totals if k not in self.ORDER]
        total = sum(self.totals.values())
        print(f'\n{"─" * 70}')
        print(f'⏱  {title}' + (f'   ({n_frames} output frames)' if n_frames else ''))
        print(f'{"─" * 70}')
        print(f'   {"step":<16}{"time":>10}{"share":>9}{"calls":>8}{"ms/call":>11}')
        for k in keys:
            t, c = self.totals[k], self.counts[k]
            pct  = 100 * t / total if total else 0.0
            print(f'   {k:<16}{t:>9.2f}s{pct:>8.1f}%{c:>8}{1000 * t / c:>10.1f}')
        print(f'   {"·" * 60}')
        line = f'   {"TOTAL":<16}{total:>9.2f}s{100.0:>8.1f}%'
        if n_frames:
            line += f'{n_frames:>8}{1000 * total / n_frames:>10.1f}  (/out frame)'
        print(line)
        print(f'{"─" * 70}')


def _track(timer: 'StepTimer | None', name: str):
    return timer.track(name) if timer is not None else nullcontext()

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 2 — SCENE-CUT DETECTION
# ─────────────────────────────────────────────────────────────────────────────
def compute_scene_cuts(frames_bgr: list[np.ndarray],
                       threshold: float = 0.30) -> tuple[list[int], list[float]]:
    boundaries = [0]
    scores: list[float] = []
    prev_hist = None
    for idx, frame in enumerate(frames_bgr):
        hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = [cv2.calcHist([hsv], [c], None, [64], [0, 256]) for c in range(3)]
        for h in hist:
            cv2.normalize(h, h)
        if prev_hist is not None:
            diffs = [cv2.compareHist(prev_hist[c], hist[c], cv2.HISTCMP_BHATTACHARYYA)
                     for c in range(3)]
            score = float(np.mean(diffs))
            scores.append(score)
            if score > threshold:
                boundaries.append(idx)
        prev_hist = hist
    return boundaries, scores

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 3 — AUTO FOREGROUND MASK
# ─────────────────────────────────────────────────────────────────────────────
def _enclosed_holes(mask_u8: np.ndarray) -> np.ndarray:
    """Binary (0/255) map of background pixels FULLY ENCLOSED by foreground.

    Pads with a 1-px background border so every background region touching the image
    edge is reached by the flood seed; whatever the flood cannot reach is background the
    foreground completely surrounds (e.g. the gap between an arm-on-hip and the torso).
    Flooding from a single corner instead would wrongly catch any background the
    foreground happens to cut off from that corner."""
    if not mask_u8.any():
        return np.zeros_like(mask_u8)
    h, w = mask_u8.shape
    padded = cv2.copyMakeBorder(mask_u8, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)
    ff = padded.copy()
    flood = np.zeros((h + 4, w + 4), np.uint8)
    cv2.floodFill(ff, flood, (0, 0), 255)      # reach all edge-connected background
    return cv2.bitwise_not(ff)[1:-1, 1:-1]     # crop pad -> enclosed holes only


def _fill_holes(mask_u8: np.ndarray, max_hole_frac: float = 1.0) -> np.ndarray:
    """Fill fully-enclosed holes in a binary (0/255) mask, up to a size limit.

    Of the enclosed holes (see _enclosed_holes), ONLY the ones whose area is
    <= max_hole_frac of the subject mask area are filled — so speckle and minor mask
    gaps are sealed, but a large genuine negative-space region (e.g. the gap between an
    arm-on-hip and the torso) is LEFT OPEN. max_hole_frac >= 1.0 fills every hole.
    """
    if not mask_u8.any():
        return mask_u8
    holes = _enclosed_holes(mask_u8)
    if max_hole_frac >= 1.0:
        return cv2.bitwise_or(mask_u8, holes)   # fill everything (original behaviour)
    max_area = max(1, int(max_hole_frac * int((mask_u8 > 0).sum())))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(holes, connectivity=8)
    small = np.zeros_like(holes)
    for lbl in range(1, n):                     # 0 = the non-hole background label
        if stats[lbl, cv2.CC_STAT_AREA] <= max_area:
            small[labels == lbl] = 255
    return cv2.bitwise_or(mask_u8, small)


def _keep_components_in_box(mask_u8: np.ndarray, x1, y1, x2, y2,
                            margin: float = 0.0) -> np.ndarray:
    """Keep only connected components that touch the (margin-expanded) box.

    Removes far-field segmentation leaks while keeping every blob belonging to the
    prompted subject. WHOLE components are kept, so the subject is never cut along
    the box edge (unlike a hard rectangular clip)."""
    if not mask_u8.any():
        return mask_u8
    h, w = mask_u8.shape
    mx, my = (x2 - x1) * margin, (y2 - y1) * margin
    bx1, by1 = max(0, int(x1 - mx)), max(0, int(y1 - my))
    bx2, by2 = min(w, int(x2 + mx)), min(h, int(y2 + my))
    n, labels = cv2.connectedComponents(mask_u8, connectivity=8)
    keep_labels = set(np.unique(labels[by1:by2, bx1:bx2]).tolist()) - {0}
    if not keep_labels:
        return np.zeros_like(mask_u8)
    return np.where(np.isin(labels, list(keep_labels)), 255, 0).astype(np.uint8)


def _drop_background_regions(add_mask: np.ndarray, ref_mask: np.ndarray,
                            frame_rgb: np.ndarray, prop_pts: list | None = None,
                            margin: float = 0.05, min_area: int = 64,
                            return_info: bool = False):
    """Keep an interactive-added region ONLY if it is clearly part of the subject.

    The interactive head is deliberately inclusive: beyond the concept silhouette
    `ref_mask` it grabs missed limbs, held props, the background a body wraps around
    (arm-on-hip <-> torso gap) AND distinct scene objects the subject merely stands
    next to (e.g. a brightly-coloured car). Only missed limbs / held props are wanted.

    For every added region we compare — via HSV hue+sat back-projection — how SUBJECT-like
    vs BACKGROUND-like its colour is, and KEEP it only when it is clearly subject-like (a
    missed limb / same clothing). Everything else is DROPPED. This is deliberately
    conservative (drop-by-default): a region that is neither clearly subject nor clearly
    background — a distinct third object such as a coloured car the person stands beside —
    is DROPPED, which is what stops background objects leaking into the mask. A region
    carrying a YOLO prop point is always kept.

    `margin` (0..1): how decisively subject-like a region must be to survive (higher =
    stricter, drops more; lower = keeps more). With `return_info`, also returns the drop
    mask and a per-region [(cx, cy, area, bg_score, subj_score, decision)] list."""
    empty = np.zeros_like(add_mask)
    if not add_mask.any() or not ref_mask.any():
        return (add_mask, empty, []) if return_info else add_mask
    extra = cv2.bitwise_and(add_mask, cv2.bitwise_not(ref_mask))
    if not extra.any():
        return (add_mask, empty, []) if return_info else add_mask

    hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
    subj_est = cv2.dilate(cv2.bitwise_or(add_mask, ref_mask),
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
    bg_sel = (subj_est == 0).astype(np.uint8)
    if int(bg_sel.sum()) < 100:                 # subject fills frame -> cannot model bg
        return (add_mask, empty, []) if return_info else add_mask
    subj_sel = (ref_mask > 0).astype(np.uint8)

    bins, ranges = [36, 50], [0, 180, 0, 256]   # HSV hue+saturation
    h_bg = cv2.calcHist([hsv], [0, 1], bg_sel, bins, ranges)
    h_subj = cv2.calcHist([hsv], [0, 1], subj_sel, bins, ranges)
    cv2.normalize(h_bg, h_bg, 0, 255, cv2.NORM_MINMAX)
    cv2.normalize(h_subj, h_subj, 0, 255, cv2.NORM_MINMAX)
    bp_bg = cv2.calcBackProject([hsv], [0, 1], h_bg, ranges, 1).astype(np.float32)
    bp_subj = cv2.calcBackProject([hsv], [0, 1], h_subj, ranges, 1).astype(np.float32)

    prop_pts = prop_pts or []
    n, labels, stats, cent = cv2.connectedComponentsWithStats(extra, connectivity=8)
    drop = np.zeros_like(add_mask)
    hh, ww = add_mask.shape
    info = []
    for lbl in range(1, n):
        area = int(stats[lbl, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        comp = labels == lbl
        is_prop = any(0 <= int(round(py)) < hh and 0 <= int(round(px)) < ww
                      and comp[int(round(py)), int(round(px))] for (px, py) in prop_pts)
        mb, ms = float(bp_bg[comp].mean()), float(bp_subj[comp].mean())
        keep = is_prop or (ms > mb + margin * 255.0)     # keep ONLY clearly subject-like
        if not keep:
            drop[comp] = 255
        if return_info:
            info.append((int(cent[lbl][0]), int(cent[lbl][1]), area, mb, ms,
                         'prop' if is_prop else ('keep' if keep else 'DROP(bg)')))
    kept = cv2.bitwise_and(add_mask, cv2.bitwise_not(drop))
    return (kept, drop, info) if return_info else kept


def _recover_held_objects(subject_mask: np.ndarray, frame_rgb: np.ndarray, boxes: list,
                          surround_frac: float = 0.6, bg_maxprob: float = 96.0,
                          max_frac: float = 0.1, min_area: int = 64) -> np.ndarray:
    """Add held objects SAM missed: non-background regions the subject surrounds.

    SAM's interactive head can omit a held object it doesn't recognise (e.g. a lav-mic
    transmitter), and YOLO may never detect it, so it is keyed out. A held object, though,
    sits AGAINST the body: it is a region the subject wraps around AND it does NOT look
    like the scene background. This scans, inside each person box, regions the current
    mask missed and adds those that are (a) surrounded by the subject on at least
    `surround_frac` of their border and (b) less background-like than `bg_maxprob`
    (0..255 HSV back-projection). Background showing through the body (matches the scene
    background) and regions the subject does not surround (open background) are left out.
    Inverse of _drop_background_regions; shares the same appearance model.
    """
    if not subject_mask.any():
        return subject_mask
    h, w = subject_mask.shape
    hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
    subj_est = cv2.dilate(subject_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
    bg_sel = (subj_est == 0).astype(np.uint8)
    if int(bg_sel.sum()) < 100:
        return subject_mask
    hbg = cv2.calcHist([hsv], [0, 1], bg_sel, [36, 50], [0, 180, 0, 256])
    cv2.normalize(hbg, hbg, 0, 255, cv2.NORM_MINMAX)
    bp_bg = cv2.calcBackProject([hsv], [0, 1], hbg, [0, 180, 0, 256], 1).astype(np.float32)
    subj_fg = subject_mask > 0
    k3 = np.ones((3, 3), np.uint8)
    add = np.zeros_like(subject_mask)
    for box in boxes:
        x1, y1, x2, y2 = (int(round(v)) for v in box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        roi = np.zeros_like(subject_mask)
        roi[y1:y2, x1:x2] = 255
        cand = cv2.bitwise_and(roi, cv2.bitwise_not(subject_mask))   # unmasked area in the box
        if not cand.any():
            continue
        n, lab, st, _cen = cv2.connectedComponentsWithStats(cand, connectivity=8)
        for L in range(1, n):
            a = int(st[L, cv2.CC_STAT_AREA])
            if a < min_area or a > max_frac * w * h:
                continue
            comp = lab == L
            cu = comp.astype(np.uint8)
            border = cv2.dilate(cu, k3) & (1 - cu)
            b = border > 0
            nb = int(b.sum())
            if nb == 0 or int((b & subj_fg).sum()) / nb < surround_frac:
                continue                                  # not surrounded -> open background
            if float(bp_bg[comp].mean()) > bg_maxprob:
                continue                                  # looks like background -> skip
            add[comp] = 255
    return cv2.bitwise_or(subject_mask, add)


def _set_detect_threshold(state, thresh):
    """Best-effort lower of SAM 3's concept-detector confidence gate so borderline
    subjects aren't dropped. No-ops (with a one-time note) if the build lacks the API."""
    if thresh is None:
        return state
    try:
        ns = sam3_predictor.set_confidence_threshold(thresh, state)
        return ns if ns is not None else state
    except Exception as e:
        if not getattr(_set_detect_threshold, '_warned', False):
            print(f'  [WARN] set_confidence_threshold unavailable ({e}); using model default gate')
            _set_detect_threshold._warned = True
        return state


def _concept_text_persons(state, mask_threshold: float, prompt: str | None = None) -> list:
    """Exhaustive SAM 3 text detection of the concept; returns [(mask_u8, bbox), ...].

    Text-prompt detection is SAM 3's best-tuned path, so it recovers clean person
    silhouettes the bare box-geometric prompt can miss. Each instance carries its bbox
    so it can be matched to a YOLO box."""
    prompt = prompt or CONCEPT_TEXT_PROMPT
    sam3_predictor.reset_all_prompts(state)
    out = sam3_predictor.set_text_prompt(prompt=prompt, state=state)
    ml = out.get('masks_logits') if isinstance(out, dict) else None
    instances = []
    if ml is None or ml.numel() == 0:
        return instances
    ml = ml.float()
    for i in range(len(ml)):
        m = ((ml[i, 0] > mask_threshold).cpu().numpy().astype(np.uint8)) * 255
        ys, xs = np.where(m > 0)
        if xs.size == 0:
            continue
        instances.append((m, (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))))
    return instances


def _bbox_iou(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return float(inter / union) if union > 0 else 0.0


def _match_concept_to_box(instances: list, box, min_iou: float):
    """Pick the text-detected concept instance whose bbox best overlaps `box`."""
    best, best_iou = None, float(min_iou)
    for m, ibox in instances:
        iou = _bbox_iou(ibox, box)
        if iou >= best_iou:
            best, best_iou = m, iou
    return best


def _fill_gaps_subtract_bg(mask: np.ndarray, frame_rgb: np.ndarray,
                           prop_pts: list | None = None, margin: float = 0.05,
                           min_area: int = 64, max_frac: float = 0.05,
                           return_info: bool = False):
    """Fill EVERY enclosed hole, then re-open only the holes that look like the background.

    An enclosed hole (a pocket the body fully surrounds) is either (a) a HELD PROP / a part
    that should be segmented — a dark mic the concept head skipped, sitting against the body —
    or (b) BACKGROUND the body wraps around: the arm-on-hip <-> torso gap, the floor between
    two dancers. Topology can't tell them apart, so we FILL them all, then — as the last,
    authoritative step — re-open only the ones whose colour matches the scene background (HSV
    hue+sat back-projection: re-open when a hole is more background-like than subject-like).
    A distinct hole (a prop) is kept as foreground; a background hole stays open. A hole with
    a prop point is always kept; a hole bigger than `max_frac` of the frame is treated as
    structural background and left open. With `return_info` also returns the re-opened mask
    and a per-hole [(cx, cy, area, bg, subj, decision)] list."""
    empty = np.zeros_like(mask)
    holes = _enclosed_holes(mask)
    if not holes.any():
        return (mask, empty, []) if return_info else mask
    filled = cv2.bitwise_or(mask, holes)

    hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
    subj_est = cv2.dilate(filled, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
    bg_sel = (subj_est == 0).astype(np.uint8)
    if int(bg_sel.sum()) < 100:                     # can't model bg -> keep everything filled
        return (filled, empty, []) if return_info else filled
    subj_sel = (mask > 0).astype(np.uint8)

    bins, ranges = [36, 50], [0, 180, 0, 256]       # HSV hue+saturation
    h_bg = cv2.calcHist([hsv], [0, 1], bg_sel, bins, ranges)
    h_sj = cv2.calcHist([hsv], [0, 1], subj_sel, bins, ranges)
    cv2.normalize(h_bg, h_bg, 0, 255, cv2.NORM_MINMAX)
    cv2.normalize(h_sj, h_sj, 0, 255, cv2.NORM_MINMAX)
    bp_bg = cv2.calcBackProject([hsv], [0, 1], h_bg, ranges, 1).astype(np.float32)
    bp_sj = cv2.calcBackProject([hsv], [0, 1], h_sj, ranges, 1).astype(np.float32)

    prop_pts = prop_pts or []
    max_area = max_frac * mask.shape[0] * mask.shape[1]
    n, labels, stats, cent = cv2.connectedComponentsWithStats(holes, connectivity=8)
    reopen = np.zeros_like(mask)
    hh, ww = mask.shape
    info = []
    for lbl in range(1, n):
        area = int(stats[lbl, cv2.CC_STAT_AREA])
        if area < min_area:
            continue                                # speckle -> stays filled
        comp = labels == lbl
        if area > max_area:                         # huge enclosed region -> structural bg
            reopen[comp] = 255
            if return_info:
                info.append((int(cent[lbl][0]), int(cent[lbl][1]), area, -1.0, -1.0, 'reopen(big)'))
            continue
        is_prop = any(0 <= int(round(py)) < hh and 0 <= int(round(px)) < ww
                      and comp[int(round(py)), int(round(px))] for (px, py) in prop_pts)
        mb, ms = float(bp_bg[comp].mean()), float(bp_sj[comp].mean())
        reopen_it = (not is_prop) and (mb > ms + margin * 255.0)
        if reopen_it:
            reopen[comp] = 255
        if return_info:
            info.append((int(cent[lbl][0]), int(cent[lbl][1]), area, mb, ms,
                         'reopen(bg)' if reopen_it else ('prop' if is_prop else 'keep')))
    final = cv2.bitwise_and(filled, cv2.bitwise_not(reopen))
    return (final, reopen, info) if return_info else final


def _select_person_boxes(cands: list, box_score_ratio: float,
                         rel_size_min: float, center_keep: float,
                         return_dropped: bool = False):
    """Choose which YOLO person boxes are real subjects vs background bystanders.

    `cands` is a list of {box, prominence, size, centrality}:
      - prominence : conf x centrality x area, the existing ranking score
      - size       : sqrt(box area) -- a LINEAR scale, so a box 'twice as small' has half
                     the size of a bigger one
      - centrality : 0..1 (1 = box centred in frame, ~0.5 = its centre at the edge)

    Two gates decide who survives:

    (1) PROMINENCE FLOOR (unchanged): keep only boxes scoring >= box_score_ratio x the
        best box's prominence -- the long-standing escape hatch (lower BOX_SCORE_RATIO to
        pull in a person being missed).

    (2) RELATIVE-SIZE BACKGROUND GATE: the reference is the LARGEST person in the frame,
        i.e. the dominant subject. A box is dropped as a background bystander only when it
        is BOTH far smaller than that subject (size / size_ref < rel_size_min, e.g. < 0.5 =
        'more than twice as small') AND off to the side (centrality < center_keep). This is
        deliberately RELATIVE:
          - a wide group where everyone is a similar (even small) size keeps everyone --
            each box ~ the reference, so no one is 'twice as small';
          - a small but CENTRED figure (a distant lead framed in the middle) is kept;
          - only a figure that is small AND pushed to the edge is removed.
        The single most-prominent box is never dropped (there is always one subject), and
        the gate is skipped entirely when rel_size_min <= 0.

    Returns the kept boxes (and, with return_dropped, also the dropped boxes)."""
    if not cands:
        return ([], []) if return_dropped else []
    best_prom = max(c['prominence'] for c in cands)
    size_ref  = max(c['size'] for c in cands) or 1.0
    top_i     = max(range(len(cands)), key=lambda i: cands[i]['prominence'])
    kept, dropped = [], []
    for i, c in enumerate(cands):
        keep_me = True
        if i != top_i:                                  # never drop the main subject
            if c['prominence'] < box_score_ratio * best_prom:
                keep_me = False                         # below the prominence floor
            elif (rel_size_min > 0 and c['size'] / size_ref < rel_size_min
                  and c['centrality'] < center_keep):
                keep_me = False                         # small AND off to the side -> bg
        (kept if keep_me else dropped).append(c['box'])
    return (kept, dropped) if return_dropped else kept


def get_foreground_mask(frame_bgr: np.ndarray, yolo_conf: float = 0.3,
                        box_score_ratio: float = 0.3,
                        mask_threshold: float = 0.5,
                        timer: 'StepTimer | None' = None,
                        extra_prop_points: list | None = None,
                        return_debug: bool = False):
    """First-frame seed mask, fusing a forceful segmenter with a topology reference.

    Per YOLO person box:
      (1) CONCEPT topology — a clean person silhouette from SAM 3's concept detector,
          taken from BOTH a box-geometric prompt AND an exhaustive text ("person")
          detection matched to the box by IoU (CONCEPT_TEXT_DETECT). The concept head is
          confidence-gated, so SAM_DETECT_THRESHOLD is lowered first to stop it dropping
          hard subjects (back turned, low contrast) YOLO did detect.
      (2) FORCEFUL interactive — sam3_model.predict_inst(box): box in, mask out, no
          concept gate, so it always fires and recovers missed limbs / held props (and a
          subject the concept head missed). Its additions are then GATED
          (INTERACTIVE_GATE_BG): only regions that look clearly like the SUBJECT are kept;
          background the body wraps around AND distinct objects the person stands beside
          (a coloured car) are dropped. YOLO prop points are always kept. The gate is
          skipped when the concept head found nothing for a box, so a fully-missed subject
          is still recovered in full.

    Enclosed holes are then size-limited-filled (MAX_HOLE_FRAC); the appearance gate is
    authoritative so recovery / hole-fill can't re-add dropped background. Returns
    (mask_u8 | None, person_boxes) — or, with return_debug=True, (mask, boxes, debug)
    whose stage masks (concept / interactive_raw / gated / dropped / final) and per-region
    decisions feed the per-shot diagnostic.
    """
    h, w = frame_bgr.shape[:2]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    dbg = ({'concept': np.zeros((h, w), np.uint8),
            'interactive_raw': np.zeros((h, w), np.uint8),
            'gated': np.zeros((h, w), np.uint8),
            'dropped': np.zeros((h, w), np.uint8),
            'final': np.zeros((h, w), np.uint8),
            'gap_kept': np.zeros((h, w), np.uint8),
            'gap_reopened': np.zeros((h, w), np.uint8),
            'decisions': [], 'gap_decisions': [],
            'person_boxes': [], 'dropped_boxes': [], 'prop_points': []}
           if return_debug else None)

    def _ret(mask, boxes):
        if dbg is not None:
            dbg['final'] = mask if mask is not None else np.zeros((h, w), np.uint8)
            return mask, boxes, dbg
        return mask, boxes

    # ── YOLO: one pass over ALL classes; split people from holdable props ──────
    with _track(timer, 'yolo'):
        yolo_results = yolo_model(frame_rgb, conf=yolo_conf, verbose=False)[0]

    person_boxes: list = []
    dropped_boxes: list = []                     # boxes the size/prominence gate removed
    prop_points: list = []                      # (x, y) pixel centres of held props
    if yolo_results.boxes is not None and len(yolo_results.boxes) > 0:
        raw_boxes = yolo_results.boxes.xyxy.cpu().numpy()
        confs     = yolo_results.boxes.conf.cpu().numpy()
        clses     = yolo_results.boxes.cls.cpu().numpy().astype(int)

        scored = []                         # per-person: prominence + size + centrality
        cx_frame, cy_frame = w / 2, h / 2
        for box, conf, cls in zip(raw_boxes, confs, clses):
            x1, y1, x2, y2 = box
            if cls == 0:                        # person -> scored candidate box
                bw, bh = x2 - x1, y2 - y1
                cx_box, cy_box = (x1 + x2) / 2, (y1 + y2) / 2
                centrality = 1.0 - (abs(cx_box - cx_frame) / cx_frame * 0.5 +
                                    abs(cy_box - cy_frame) / cy_frame * 0.5)
                area_frac = (bw * bh) / (w * h)
                prominence = float(conf) * (1 + centrality) * (1 + area_frac * 3)
                scored.append({'box': box, 'prominence': prominence,
                               'size': float(np.sqrt(max(bw, 1.0) * max(bh, 1.0))),
                               'centrality': float(centrality)})
            elif (MASK_HELD_OBJECTS and cls in HELD_OBJECT_CLASSES and
                  (x2 - x1) * (y2 - y1) <= HELD_OBJECT_MAX_FRAC * w * h):
                prop_points.append(((x1 + x2) / 2, (y1 + y2) / 2))   # held-prop point

        # prominence floor (existing) + relative-size background gate (drops a person who
        # is BOTH much smaller than the biggest subject AND off to the side).
        if scored:
            person_boxes, dropped_boxes = _select_person_boxes(
                scored, box_score_ratio, PERSON_REL_SIZE_MIN, PERSON_CENTER_KEEP,
                return_dropped=True)

    # explicit prop points (pixel coords) for props YOLO can't detect — treated
    # exactly like a detected prop: fed to the interactive head and always kept.
    if extra_prop_points:
        prop_points.extend(extra_prop_points)

    if dbg is not None:
        dbg['person_boxes'] = person_boxes
        dbg['dropped_boxes'] = dropped_boxes
        dbg['prop_points'] = prop_points

    # ── SAM 3 — concept topology (clean) fused with forceful interactive head ──
    with _track(timer, 'sam'), torch.no_grad(), \
         torch.autocast(device_type='cuda', dtype=torch.bfloat16):

        if person_boxes:
            state = sam3_predictor.set_image(Image.fromarray(frame_rgb))
            state = _set_detect_threshold(state, SAM_DETECT_THRESHOLD)

            # exhaustive text detection once per frame; matched to each box below
            concept_instances = (_concept_text_persons(state, mask_threshold)
                                 if CONCEPT_TEXT_DETECT else [])

            combined_mask = np.zeros((h, w), dtype=np.uint8)
            dropped_bg = np.zeros((h, w), dtype=np.uint8)   # background the gate removed
            for box in person_boxes:
                x1, y1, x2, y2 = (float(v) for v in box)

                # (1) CONCEPT topology — box-geometric prompt UNION matched text instance
                concept_box = np.zeros((h, w), dtype=np.uint8)
                box_norm = [((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h,
                            (x2 - x1) / w, (y2 - y1) / h]
                sam3_predictor.reset_all_prompts(state)
                out = sam3_predictor.add_geometric_prompt(box=box_norm, label=True,
                                                          state=state)
                ml = out['masks_logits']
                if ml is not None and ml.numel() > 0:
                    sc = out['scores'].float()
                    concept_box = cv2.bitwise_or(concept_box,
                        ((ml[int(torch.argmax(sc)), 0].float() > mask_threshold)
                         .cpu().numpy().astype(np.uint8)) * 255)
                matched = _match_concept_to_box(concept_instances,
                                                (x1, y1, x2, y2), CONCEPT_MATCH_IOU)
                if matched is not None:
                    concept_box = cv2.bitwise_or(concept_box, matched)
                combined_mask = cv2.bitwise_or(combined_mask, concept_box)
                if dbg is not None:
                    dbg['concept'] = cv2.bitwise_or(dbg['concept'], concept_box)

                # (2) FORCEFUL interactive head — always fires; recovers missed limbs /
                #     props, then gated to keep only clearly-subject additions.
                if FORCE_INTERACTIVE:
                    pts = [(px, py) for (px, py) in prop_points
                           if x1 <= px <= x2 and y1 <= py <= y2]
                    pc = np.array(pts, dtype=np.float32) if pts else None
                    pl = np.ones(len(pts), dtype=np.int32) if pts else None
                    masks, ious, _low = sam3_model.predict_inst(
                        state, point_coords=pc, point_labels=pl,
                        box=np.array([x1, y1, x2, y2], dtype=np.float32),
                        multimask_output=True, return_logits=True)
                    forced = (masks[int(np.argmax(ious))] > INTERACTIVE_MASK_THRESHOLD
                              ).astype(np.uint8) * 255
                    forced = _keep_components_in_box(forced, x1, y1, x2, y2,
                                                     BOX_CLIP_MARGIN)
                    if dbg is not None:
                        dbg['interactive_raw'] = cv2.bitwise_or(dbg['interactive_raw'], forced)
                    # keep only clearly-subject additions; drop bg AND distinct objects.
                    # skipped if concept found nothing for this box (recover subject in full)
                    if INTERACTIVE_GATE_BG and concept_box.any():
                        if return_debug:
                            gated, drp, decs = _drop_background_regions(
                                forced, concept_box, frame_rgb, prop_pts=pts,
                                margin=GATE_BG_MARGIN, return_info=True)
                            dbg['decisions'].extend(decs)
                        else:
                            gated = _drop_background_regions(
                                forced, concept_box, frame_rgb, prop_pts=pts,
                                margin=GATE_BG_MARGIN)
                            drp = cv2.bitwise_and(forced, cv2.bitwise_not(gated))
                        dropped_bg = cv2.bitwise_or(dropped_bg, drp)
                        if dbg is not None:
                            dbg['gated'] = cv2.bitwise_or(dbg['gated'], gated)
                            dbg['dropped'] = cv2.bitwise_or(dbg['dropped'], drp)
                        forced = gated
                    elif dbg is not None:
                        dbg['gated'] = cv2.bitwise_or(dbg['gated'], forced)
                    combined_mask = cv2.bitwise_or(combined_mask, forced)

            if RECOVER_HELD_OBJECTS:
                combined_mask = _recover_held_objects(
                    combined_mask, frame_rgb, person_boxes,
                    surround_frac=RECOVER_SURROUND_FRAC, bg_maxprob=RECOVER_BG_MAXPROB,
                    max_frac=RECOVER_MAX_FRAC)
            # GAP FILL — fill every enclosed hole, then (LAST step) re-open only the
            # holes whose colour matches the background: a held prop / dark region the
            # heads skipped stays filled, a background gap the body wraps stays open.
            pre_gap = combined_mask.copy()
            if GAP_FILL:
                if return_debug:
                    combined_mask, gap_reopen, gap_info = _fill_gaps_subtract_bg(
                        combined_mask, frame_rgb, prop_pts=prop_points, margin=GAP_BG_MARGIN,
                        min_area=GAP_MIN_AREA, max_frac=GAP_MAX_FRAC, return_info=True)
                    dbg['gap_reopened'] = gap_reopen
                    dbg['gap_kept'] = cv2.bitwise_and(combined_mask, cv2.bitwise_not(pre_gap))
                    dbg['gap_decisions'] = gap_info
                else:
                    combined_mask = _fill_gaps_subtract_bg(
                        combined_mask, frame_rgb, prop_pts=prop_points, margin=GAP_BG_MARGIN,
                        min_area=GAP_MIN_AREA, max_frac=GAP_MAX_FRAC)
            elif FILL_MASK_HOLES:
                combined_mask = _fill_holes(combined_mask, MAX_HOLE_FRAC)
            # the conservative OUTWARD gate is authoritative: never let gap-fill re-add
            # the distinct background it dropped (a car the person stands beside)
            if dropped_bg.any():
                combined_mask = cv2.bitwise_and(combined_mask, cv2.bitwise_not(dropped_bg))
            return _ret(combined_mask if combined_mask.any() else None, person_boxes)

        # ── Fallback: YOLO found no person -> SAM 3 text-prompt concept path ────
        print('  [WARN] No person detected by YOLO — using SAM 3 text-prompt fallback')
        state = sam3_predictor.set_image(Image.fromarray(frame_rgb))
        state = _set_detect_threshold(state, SAM_DETECT_THRESHOLD)
        sam3_predictor.reset_all_prompts(state)
        out = sam3_predictor.set_text_prompt(prompt=CONCEPT_TEXT_PROMPT, state=state)
        masks_logits = out['masks_logits']
        if masks_logits is None or masks_logits.numel() == 0:
            return _ret(None, [])
        masks_logits = masks_logits.float()
        combined_mask = np.zeros((h, w), dtype=np.uint8)
        for mi in range(len(masks_logits)):
            m = ((masks_logits[mi, 0] > mask_threshold)
                 .cpu().numpy().astype(np.uint8)) * 255
            combined_mask = cv2.bitwise_or(combined_mask, m)
        if dbg is not None:
            dbg['concept'] = cv2.bitwise_or(dbg['concept'], combined_mask)
        if FILL_MASK_HOLES:
            combined_mask = _fill_holes(combined_mask, MAX_HOLE_FRAC)
        return _ret(combined_mask if combined_mask.any() else None, [])

# ─────────────────────────────────────────────────────────────────────────────
#  Small reusable helpers
# ─────────────────────────────────────────────────────────────────────────────
def make_seed_overlay(frame_bgr: np.ndarray, mask: np.ndarray, alpha: float = 0.45,
                      boxes: list | None = None, box_color=(255, 220, 0),
                      box_thickness: int = 2) -> np.ndarray:
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    m   = mask > 127
    rgb[m] = rgb[m] * (1 - alpha) + np.array([255, 40, 40], np.float32) * alpha
    out = np.clip(rgb, 0, 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (255, 255, 255), 2)
    if boxes:
        for box in boxes:
            x1, y1, x2, y2 = (int(round(v)) for v in box)
            cv2.rectangle(out, (x1, y1), (x2, y2), box_color, box_thickness)
    return out


def composite_result(pha_u8: np.ndarray, fgr_u8: np.ndarray,
                     mode: str, green: np.ndarray) -> np.ndarray:
    pha = pha_u8 / 255.
    fgr = fgr_u8 / 255.
    if mode == 'greenscreen':
        comp = fgr + green * (1.0 - pha)
    elif mode == 'alpha':
        comp = np.repeat(pha, 3, axis=2)
    else:
        comp = fgr
    return np.round(np.clip(comp * 255, 0, 255)).astype(np.uint8)


def thumbnail(img: np.ndarray, max_w: int = 640) -> np.ndarray:
    h, w = img.shape[:2]
    if w <= max_w:
        return img
    return cv2.resize(img, (max_w, int(h * max_w / w)), interpolation=cv2.INTER_AREA)

# ─────────────────────────────────────────────────────────────────────────────
#  STEP 4 — MATANYONE 2 SHOT PROCESSOR
# ─────────────────────────────────────────────────────────────────────────────
def run_matanyone2_on_shot(
    frames_bgr: list[np.ndarray],
    first_frame_mask: np.ndarray,
    n_warmup_static: int = 20,
    n_warmup_motion: int = 20,
    r_erode: int = 0,
    r_dilate: int = 20,
    timer: 'StepTimer | None' = None,
) -> list[dict]:
    with _track(timer, 'matting_pre'):
        processor = InferenceCore(matanyone2_model, cfg=matanyone2_model.cfg)

        vframes = torch.stack([
            torch.from_numpy(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)
            for f in frames_bgr
        ])  # uint8 [T, C, H, W]

        h_orig, w_orig = vframes.shape[-2:]
        resized = False
        if MAX_SIZE > 0 and min(h_orig, w_orig) > MAX_SIZE:
            scale = MAX_SIZE / min(h_orig, w_orig)
            new_h, new_w = int(h_orig * scale), int(w_orig * scale)
            vframes = (F.interpolate(vframes.float(), size=(new_h, new_w), mode='area')
                       .round().clamp_(0, 255).to(torch.uint8))
            resized = True
        h, w = vframes.shape[-2:]

        warmup_static = vframes[0:1].repeat(n_warmup_static, 1, 1, 1)
        n_motion_actual = min(n_warmup_motion, len(frames_bgr))
        warmup_motion = vframes[:n_motion_actual]

        vframes_full = torch.cat([warmup_static, warmup_motion, vframes], dim=0)
        total_warmup = n_warmup_static + n_motion_actual
        total_len    = len(vframes_full)

        mask_np = first_frame_mask.copy()
        if r_dilate > 0:
            mask_np = gen_dilate(mask_np, r_dilate, r_dilate)
        if r_erode > 0:
            mask_np = gen_erosion(mask_np, r_erode, r_erode)

        mask_t = torch.from_numpy(mask_np).float().to(device)
        if resized:
            mask_t = F.interpolate(mask_t.unsqueeze(0).unsqueeze(0),
                                   size=(h, w), mode='nearest')[0, 0]

    objects = [1]
    results = []

    with torch.inference_mode():
        for ti in range(total_len):
            is_static_warmup = ti < n_warmup_static
            is_warmup        = ti < total_warmup
            step_name = 'matting_warmup' if is_warmup else 'matting_infer'

            with _track(timer, step_name):
                image = (vframes_full[ti].float() / 255.).to(device)

                if ti == 0:
                    out_prob = processor.step(image, mask_t, objects=objects)
                    out_prob = processor.step(image, first_frame_pred=True)
                elif is_static_warmup:
                    out_prob = processor.step(image, first_frame_pred=True)
                else:
                    out_prob = processor.step(image)

                alpha_t = processor.output_prob_to_mask(out_prob)

            if not is_warmup:
                with _track(timer, 'matting_post'):
                    image_np = vframes_full[ti].permute(1, 2, 0).numpy()
                    pha_np   = alpha_t.unsqueeze(2).cpu().float().numpy()
                    fgr_np   = (image_np / 255.) * pha_np

                    pha_out = np.round(np.clip(pha_np * 255, 0, 255)).astype(np.uint8)
                    fgr_out = np.round(np.clip(fgr_np * 255, 0, 255)).astype(np.uint8)

                    if resized:
                        pha_out = cv2.resize(pha_out[..., 0], (w_orig, h_orig),
                                             interpolation=cv2.INTER_LINEAR)[..., None]
                        fgr_out = cv2.resize(fgr_out, (w_orig, h_orig),
                                             interpolation=cv2.INTER_LINEAR)

                    results.append({'pha': pha_out, 'fgr': fgr_out})

    return results


print('✅ Pipeline functions defined')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  FULL PIPELINE  (batch over every video in INPUT_DIR)
# ─────────────────────────────────────────────────────────────────────────────
import gc, glob

all_videos = []
for ext in VIDEO_EXTENSIONS:
    all_videos.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
all_videos = sorted(all_videos)
print(f'Found {len(all_videos)} video(s) to process')

grand_timer   = StepTimer()
grand_frames  = 0
preview_shots = []
preview_shots_by_video = {}      # base_name -> per-shot previews, for the by-index diagnostic
last_results  = []

for video_idx, INPUT_VIDEO in enumerate(all_videos):
    base_name   = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
    output_path = os.path.join(OUTPUT_DIR, f'{base_name}_matte.mp4')
    alpha_path  = os.path.join(OUTPUT_DIR, f'{base_name}_alpha.mp4')
    seed_dir    = os.path.join(OUTPUT_DIR, f'{base_name}_seed_masks')

    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f'\n[{video_idx+1}/{len(all_videos)}] Skipping (already done): {base_name}')
        continue

    print(f'\n{"="*70}\n[{video_idx+1}/{len(all_videos)}] Processing: {base_name}\n{"="*70}')
    timer = StepTimer()
    preview_shots = []

    try:
        cap = cv2.VideoCapture(INPUT_VIDEO)
        fps    = cap.get(cv2.CAP_PROP_FPS)
        total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        print(f'Video: {total} frames @ {fps:.2f}fps  |  {width}×{height}')

        print('\n[1/5] Decoding frames ...')
        with timer.track('decode'):
            cap = cv2.VideoCapture(INPUT_VIDEO)
            all_frames = []
            pbar = tqdm(total=total if total > 0 else None, desc='Decoding')
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                all_frames.append(frame)
                pbar.update(1)
            pbar.close()
            cap.release()
        total = len(all_frames)
        print(f'Decoded {total} frames')

        print('\n[2/5] Detecting scene cuts ...')
        with timer.track('scene_cut'):
            cut_starts, _ = compute_scene_cuts(all_frames, threshold=CUT_THRESHOLD)
        shot_intervals = [
            (s, cut_starts[i + 1] if i + 1 < len(cut_starts) else total)
            for i, s in enumerate(cut_starts)
        ]
        print(f'Found {len(shot_intervals)} shot(s): {shot_intervals}')

        print('\n[3/5] Segmenting + matting shots ...')
        if SAVE_SEED_MASKS:
            os.makedirs(seed_dir, exist_ok=True)
        all_results = []

        for shot_idx, (shot_start, shot_end) in enumerate(shot_intervals):
            shot_frames = all_frames[shot_start:shot_end]
            n_frames    = len(shot_frames)
            print(f'\n  Shot {shot_idx+1}/{len(shot_intervals)}: '
                  f'frames {shot_start}–{shot_end-1} ({n_frames} frames)')
            if n_frames == 0:
                continue

            _mp = MANUAL_PROP_POINTS.get(base_name, {}).get(shot_idx + 1, [])
            _h0, _w0 = shot_frames[0].shape[:2]
            _extra_pp = [(fx * _w0, fy * _h0) for (fx, fy) in _mp]
            if _extra_pp:
                print(f'  Applying {len(_extra_pp)} manual prop point(s) to shot {shot_idx+1} seed')
            first_frame_mask, yolo_boxes = get_foreground_mask(
                shot_frames[0], yolo_conf=YOLO_CONF,
                box_score_ratio=BOX_SCORE_RATIO,
                mask_threshold=SAM_MASK_THRESHOLD,
                extra_prop_points=_extra_pp,
                timer=timer)

            if first_frame_mask is None:
                print(f'  [WARN] No foreground found — writing blank alpha for shot {shot_idx+1}.')
                h, w = shot_frames[0].shape[:2]
                all_results.extend({'pha': np.zeros((h, w, 1), np.uint8),
                                    'fgr': np.zeros((h, w, 3), np.uint8)} for _ in shot_frames)
                continue

            with timer.track('mask_debug'):
                overlay = make_seed_overlay(shot_frames[0], first_frame_mask, boxes=yolo_boxes)
                if SAVE_SEED_MASKS:
                    cv2.imwrite(os.path.join(seed_dir, f'shot_{shot_idx+1:02d}_mask.png'),
                                first_frame_mask)
                    cv2.imwrite(os.path.join(seed_dir, f'shot_{shot_idx+1:02d}_overlay.png'),
                                cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

            shot_results = run_matanyone2_on_shot(
                shot_frames, first_frame_mask,
                n_warmup_static=N_WARMUP_STATIC, n_warmup_motion=N_WARMUP_MOTION,
                r_erode=R_ERODE, r_dilate=R_DILATE, timer=timer)
            all_results.extend(shot_results)

            matted0 = composite_result(shot_results[0]['pha'], shot_results[0]['fgr'],
                                       'greenscreen', GREEN)
            matted_last = composite_result(shot_results[-1]['pha'], shot_results[-1]['fgr'],
                                           'greenscreen', GREEN)
            preview_shots.append({'start': shot_start, 'end': shot_end,
                                  'overlay': thumbnail(overlay), 'matted': thumbnail(matted0),
                                  'matted_last': thumbnail(matted_last),
                                  'n_boxes': len(yolo_boxes)})

            torch.cuda.empty_cache(); gc.collect()

        print(f'\nProcessed {len(all_results)} output frames total')
        preview_shots_by_video[base_name] = preview_shots   # cache for the by-index diagnostic (cell below)

        print('\n[4/5] Compositing ...')
        composite_frames, alpha_frames = [], []
        with timer.track('composite'):
            for r in tqdm(all_results, desc='Compositing'):
                composite_frames.append(composite_result(r['pha'], r['fgr'], OUTPUT_MODE, GREEN))
                alpha_frames.append(r['pha'])

        print('\n[5/5] Encoding ...')
        enc = dict(codec='libx264', fps=fps,
                   output_params=['-preset', ENCODER_PRESET, '-crf', str(ENCODER_CRF)])
        tmp_output, tmp_alpha = output_path + '.tmp.mp4', alpha_path + '.tmp.mp4'
        with timer.track('encode'):
            imageio.mimwrite(tmp_output, composite_frames, **enc)
            os.rename(tmp_output, output_path)

            if WRITE_ALPHA_VIDEO:
                imageio.mimwrite(tmp_alpha, [a[..., 0] for a in alpha_frames], **enc)
                os.rename(tmp_alpha, alpha_path)

            if OUTPUT_MODE == 'transparent':
                rgba_dir = os.path.join(OUTPUT_DIR, f'{base_name}_rgba_frames')
                os.makedirs(rgba_dir, exist_ok=True)
                for fi, (rgb, a) in enumerate(zip(composite_frames, alpha_frames)):
                    Image.fromarray(np.concatenate([rgb, a], axis=2)).save(
                        os.path.join(rgba_dir, f'{fi:05d}.png'))
                print(f'RGBA frames saved to {rgba_dir}')

        print(f'\n✅ Done: {base_name}')
        print(f'   Composite : {output_path}')
        if WRITE_ALPHA_VIDEO:
            print(f'   Alpha     : {alpha_path}')
        if SAVE_SEED_MASKS:
            print(f'   Seed masks: {seed_dir}')

        n_out = len(all_results)
        timer.report(title=base_name, n_frames=n_out)
        grand_timer.merge(timer); grand_frames += n_out

        last_results = all_results
        last_shot_intervals = shot_intervals

        del all_frames, composite_frames, alpha_frames
        torch.cuda.empty_cache(); gc.collect()

    except Exception as e:
        print(f'\n❌ FAILED: {base_name} — {e}\n   Continuing to next video ...')
        for tmp in [output_path + '.tmp.mp4', alpha_path + '.tmp.mp4']:
            if os.path.exists(tmp):
                os.remove(tmp)
        torch.cuda.empty_cache(); gc.collect()
        continue

print(f'\n{"="*70}\nAll videos processed.')

if grand_timer.totals:
    grand_timer.report(title='ALL VIDEOS — grand total', n_frames=grand_frames)

In [ ]:
import matplotlib.pyplot as plt

# ──── DIAGNOSTIC · per-shot SEED + matte, for ANY processed video by index ────
# For the chosen video, each row shows a shot's seed (YOLO boxes + SAM 3 mask) beside its
# matte on the FIRST and LAST frame:
#   - background that appears only on the LAST frame = MatAnyone drift (memory latched onto
#     the background), NOT a seed error -> lower R_DILATE / add R_ERODE.
#   - a whole extra person already in the seed = a YOLO / size-gate issue -> tune
#     PERSON_REL_SIZE_MIN / PERSON_CENTER_KEEP / BOX_SCORE_RATIO (see the per-shot seed
#     diagnostic below for which boxes the size gate kept vs dropped).
# Pick the video with DIAG_PREVIEW_INDEX (0-based index into INPUT_DIR, the same order the
# pipeline processed). It reuses the mattes the pipeline already produced, so every video
# from the last run is viewable here by index -- not only the most recent one.

DIAG_PREVIEW_INDEX = 6     # which video in INPUT_DIR to inspect

_names = [os.path.splitext(os.path.basename(v))[0] for v in all_videos]
_cache = globals().get('preview_shots_by_video', {})

if not _cache:
    print('No previews cached yet — run the full pipeline cell first.')
elif not (0 <= DIAG_PREVIEW_INDEX < len(_names)) or _names[DIAG_PREVIEW_INDEX] not in _cache:
    print(f'Video index {DIAG_PREVIEW_INDEX} has no cached preview '
          '(not processed this run, or skipped as already-done).')
    print('Available  (index : video):')
    for _i, _n in enumerate(_names):
        if _n in _cache:
            print(f'   {_i} : {_n}  ({len(_cache[_n])} shot(s))')
else:
    base  = _names[DIAG_PREVIEW_INDEX]
    shots = _cache[base]
    print(f'Preview for [{DIAG_PREVIEW_INDEX}] {base} — {len(shots)} shot(s)')
    if not shots:
        print('   (no shots — video produced no foreground seed)')
    else:
        n = len(shots)
        fig, axes = plt.subplots(n, 3, figsize=(16, 4.2 * n))
        axes = np.array(axes).reshape(n, 3)
        for i, sh in enumerate(shots):
            axes[i, 0].imshow(sh['overlay'])
            axes[i, 0].set_title(
                f"Shot {i+1} (frames {sh['start']}–{sh['end']-1})  ·  "
                f"{sh.get('n_boxes', 0)} YOLO box(es) + SAM 3 seed", fontsize=9)
            axes[i, 1].imshow(sh['matted'])
            axes[i, 1].set_title(f"Shot {i+1}  ·  matte @ FIRST frame", fontsize=9)
            axes[i, 2].imshow(sh['matted_last'])
            axes[i, 2].set_title(f"Shot {i+1}  ·  matte @ LAST frame (drift check)", fontsize=9)
            for c in range(3):
                axes[i, c].axis('off')
        plt.suptitle('Seed (left) vs matte at FIRST (middle) and LAST (right) frame of each '
                     'shot.  Background that shows up only on the right = MatAnyone drift, '
                     'not a seed error — lower R_DILATE / add R_ERODE.', fontsize=12)
        plt.tight_layout()
        out_png = os.path.join(OUTPUT_DIR, f'shot_preview_{base}.png')
        plt.savefig(out_png, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'Shot preview saved -> {out_png}')


In [ ]:
from google.colab import files

print(f'Downloading: {output_path}')
files.download(output_path)

In [ ]:
import matplotlib.pyplot as plt

# ── Pick which video to inspect ───────────────────────────────────────────────
# Defaults to the first video found in INPUT_DIR. Change the index or set an
# explicit path (e.g. INSPECT_VIDEO = '/content/videos/mv.mp4') to inspect another.
INSPECT_VIDEO = all_videos[4] if all_videos else INPUT_VIDEO
print(f'Inspecting: {INSPECT_VIDEO}')

cap = cv2.VideoCapture(INSPECT_VIDEO)
frames_for_plot = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames_for_plot.append(frame)
cap.release()
print(f'Decoded {len(frames_for_plot)} frames')

cut_starts, scores = compute_scene_cuts(frames_for_plot, threshold=CUT_THRESHOLD)

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(scores, linewidth=0.8, color='steelblue', label='HSV histogram diff')
ax.axhline(CUT_THRESHOLD, color='crimson', linestyle='--', linewidth=1.2,
           label=f'Current threshold ({CUT_THRESHOLD})')
for c in cut_starts[1:]:   # skip the implicit 0-boundary
    ax.axvline(c, color='orange', linewidth=0.6, alpha=0.6)
ax.set_xlabel('Frame')
ax.set_ylabel('Bhattacharyya distance')
ax.set_title(f'Scene cut signal — {len(cut_starts)} shot(s) detected   |   {os.path.basename(INSPECT_VIDEO)}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Cut boundaries at threshold {CUT_THRESHOLD}: {cut_starts}')


In [ ]:
# ──── DIAGNOSTIC · per-SHOT seed stages (concept / interactive / gate / final) ────
# Runs the REAL seed logic (get_foreground_mask) on the FIRST FRAME OF EVERY SHOT of the
# chosen video, so you can see per shot exactly where the mask comes from and which stage
# is responsible for a leak. One row per shot, five panels:
#   1 concept silhouette   2 interactive RAW (before gate)   3 gate decision
#   (red = kept as subject, blue = dropped as background/other)   4 FINAL seed
#   5 seed + dilate (what matting receives).
# Per interactive-added region it prints bg-likeness vs subject-likeness + the decision so
# you can tune GATE_BG_MARGIN. Uses get_foreground_mask directly, so it always matches the
# pipeline exactly (no drift).
import glob
import matplotlib.pyplot as plt

_vids = sorted(sum([glob.glob(os.path.join(INPUT_DIR, e)) for e in VIDEO_EXTENSIONS], []))
assert _vids, f'no videos in {INPUT_DIR}'
DIAG_VIDEO_INDEX = 5                    # which video in INPUT_DIR to inspect
DIAG_VIDEO = _vids[DIAG_VIDEO_INDEX]
DIAG_BASE = os.path.splitext(os.path.basename(DIAG_VIDEO))[0]
print('Per-shot seed diagnostic for:', os.path.basename(DIAG_VIDEO))

cap = cv2.VideoCapture(DIAG_VIDEO)
_frames = []
while True:
    ok, fr = cap.read()
    if not ok:
        break
    _frames.append(fr)
cap.release()
assert _frames, 'no frames decoded'

cut_starts, _ = compute_scene_cuts(_frames, threshold=CUT_THRESHOLD)
shots = [(s, cut_starts[i + 1] if i + 1 < len(cut_starts) else len(_frames))
         for i, s in enumerate(cut_starts)]
print(f'{len(shots)} shot(s): {shots}')

def _ov(frame_rgb, mask, color=(255, 40, 40), a=0.5):
    o = frame_rgb.astype(np.float32).copy(); sel = mask > 127
    o[sel] = o[sel] * (1 - a) + np.array(color, np.float32) * a
    return np.clip(o, 0, 255).astype(np.uint8)

rows = []
for si, (a, b) in enumerate(shots):
    f0 = _frames[a]
    rgb = cv2.cvtColor(f0, cv2.COLOR_BGR2RGB)
    _mp = MANUAL_PROP_POINTS.get(DIAG_BASE, {}).get(si + 1, [])
    _epp = [(fx * f0.shape[1], fy * f0.shape[0]) for (fx, fy) in _mp]
    seed, boxes, d = get_foreground_mask(
        f0, yolo_conf=YOLO_CONF, box_score_ratio=BOX_SCORE_RATIO,
        mask_threshold=SAM_MASK_THRESHOLD, extra_prop_points=_epp, return_debug=True)
    final = d['final']
    try:
        dil = gen_dilate(final.copy(), R_DILATE, R_DILATE) if R_DILATE > 0 else final.copy()
    except Exception:
        dil = (cv2.dilate(final, np.ones((max(1, R_DILATE), max(1, R_DILATE)), np.uint8))
               if R_DILATE > 0 else final.copy())

    # gate-decision panel: red = kept (subject) extra, blue = dropped (bg / other)
    dec = rgb.astype(np.float32).copy()
    kept_extra = cv2.bitwise_and(d['gated'], cv2.bitwise_not(d['concept']))
    dec[kept_extra > 127] = dec[kept_extra > 127] * 0.4 + np.array([255, 40, 40], np.float32) * 0.6
    dec[d['dropped'] > 127] = dec[d['dropped'] > 127] * 0.4 + np.array([40, 90, 255], np.float32) * 0.6
    dec = np.clip(dec, 0, 255).astype(np.uint8)

    rows.append((si, a, b, len(boxes), rgb, d, final, dil, dec, _epp))
    gk_px = int((d.get('gap_kept', np.zeros_like(final)) > 0).sum())
    gr_px = int((d.get('gap_reopened', np.zeros_like(final)) > 0).sum())
    print(f'\nShot {si+1} (frames {a}-{b-1}): {len(boxes)} person box(es), '
          f'{len(d["prop_points"])} prop pt(s)  |  gap fill: kept {gk_px}px '
          f'(prop/should-segment), re-opened {gr_px}px (background)')
    if not d['decisions']:
        print('    (no interactive-added regions to judge)')
    for (cx, cy, ar, mb, ms, dcn) in d['decisions']:
        print(f'    extra @({cx:>4},{cy:>4}) area={ar:>7}  bg={mb:6.1f}  subj={ms:6.1f}  -> {dcn}')
    for (cx, cy, ar, mb, ms, dcn) in d.get('gap_decisions', []):
        print(f'    gap   @({cx:>4},{cy:>4}) area={ar:>7}  bg={mb:6.1f}  subj={ms:6.1f}  -> {dcn}')

n = len(rows)
fig, axes = plt.subplots(n, 6, figsize=(34, 6 * n))
axes = np.array(axes).reshape(n, 6)
titles = ['1. concept', '2. interactive RAW', '3. gate (red=kept, blue=dropped)',
          '4. FINAL seed', f'5. seed + dilate {R_DILATE}px',
          '6. gaps (green=kept prop, magenta=re-opened bg)']
for r, (si, a, b, nb, rgb, d, final, dil, dec, epp) in enumerate(rows):
    gap_img = rgb.astype(np.float32).copy()
    gk, gr = d.get('gap_kept'), d.get('gap_reopened')
    if gk is not None:
        gap_img[gk > 127] = gap_img[gk > 127] * 0.4 + np.array([40, 230, 90], np.float32) * 0.6
    if gr is not None:
        gap_img[gr > 127] = gap_img[gr > 127] * 0.4 + np.array([230, 40, 230], np.float32) * 0.6
    gap_img = np.clip(gap_img, 0, 255).astype(np.uint8)
    imgs = [_ov(rgb, d['concept']), _ov(rgb, d['interactive_raw']), dec,
            _ov(rgb, final), _ov(rgb, dil), gap_img]
    for c in range(6):
        axes[r, c].imshow(imgs[c]); axes[r, c].axis('off')
        axes[r, c].set_title(f'Shot {si+1} ({nb} box) · {titles[c]}', fontsize=8)
        if epp:                                  # yellow x = manual prop point
            axes[r, c].scatter([p[0] for p in epp], [p[1] for p in epp],
                               s=70, marker='x', c='yellow', linewidths=2)
plt.suptitle('Per-shot seed stages.  Missing in col 2 -> interactive never captured it.  '
             'Blue in col 3 -> gate dropped it as background/other.', fontsize=12)
plt.tight_layout()
out_png = os.path.join(OUTPUT_DIR, 'seed_diagnostic_per_shot.png')
plt.savefig(out_png, dpi=110, bbox_inches='tight'); plt.show()
print('\nsaved ->', out_png)